#  Economic Integration: Sahel Security Analysis
---
This notebook investigates the relationship between armed conflict and food prices
in the Sahel region (Burkina Faso, Mali, Niger).

**Core question:**
> Does a spike in conflict activity in a given zone precede a rise in food prices
> in that same zone in the following weeks?

**Data sources:**
- **ACLED**: Armed conflict events (already processed)
- **WFP/HDX**: Food market prices (mil, sorghum, maize) by market and region

**Method:**
1. Download WFP food price data via HDX API
2. Clean and align with ACLED geographic/temporal structure
3. Join on Admin1 + Year/Month
4. Compute lagged correlations (conflict at T → prices at T+1, T+2, T+4 weeks)
5. Visualize the impact

**Key join variable:** `admin1` + `year_month`

In [1]:
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# Load already processed ACLED data
df_acled = pd.read_csv("../data/processed/acled_processed.csv", parse_dates=["event_date"])
df_acled["year_month"] = df_acled["event_date"].dt.to_period("M").astype(str)

print(f"ACLED records loaded : {len(df_acled):,}")
print(f"Period               : {df_acled['event_date'].min().date()} -> {df_acled['event_date'].max().date()}")
print(f"Countries            : {list(df_acled['country'].unique())}")
print(f"Admin1 regions       : {df_acled['admin1'].nunique()} unique regions")

ACLED records loaded : 23,156
Period               : 2020-01-01 -> 2025-03-28
Countries            : ['Niger', 'Burkina Faso', 'Mali']
Admin1 regions       : 31 unique regions


## 1. Data Source Selection: Methodology & Decisions
---

### Initial approach: WFP/HDX API
Our first attempt was to use the **WFP Global Food Prices** dataset via the
HDX (Humanitarian Data Exchange) API, which theoretically provides market-level
food prices (millet, sorghum, maize) for Burkina Faso, Mali, and Niger.

**Problem encountered:** The HDX API returned `success: False` on all requests,
despite the resource ID being valid. We fell back to a direct CSV download which
worked, but revealed a critical limitation:

>  **WFP data covers only up to 2021**: leaving only 24 months of overlap
> with our ACLED conflict data (2020–2025). This is insufficient for robust
> statistical analysis.

---

### Second attempt: World Bank & JMR datasets
We explored the World Bank Real-Time Food Prices (RTFP) and the
Joint Monitoring Report (JMR) dataset, which combines conflict and food
price data in a single source.

**Problem encountered:** Using a dataset that already integrates ACLED conflict
data as an input to analyze the relationship between conflict and food prices
would constitute **circular reasoning**, using a conclusion to prove itself.
This approach was discarded on methodological grounds.

---

### Final choice: IMF World Economic Outlook (WEO)
We selected the **IMF DataMapper API** as our primary economic data source for
the following reasons:

| Criterion | Assessment |
|---|---|
| **Independence** | Fully independent from ACLED — no circular reasoning risk |
| **Coverage** | 2018–2024 actual data for Burkina Faso, Mali, Niger |
| **Reliability** | IMF is the institutional reference for macroeconomic data |
| **Accessibility** | Free, stable REST API, no authentication required |
| **Granularity** | Annual inflation rate (PCPIPCH indicator) per country |

**Indicator used:** `PCPIPCH`: Inflation rate, average consumer prices (annual % change)

>  **Known limitation:** The IMF provides annual country-level data, not
> monthly or sub-national data. This means our join with ACLED will be at the
> **country × year** level, not admin1 × month as originally planned.
> This is a deliberate tradeoff: we sacrifice granularity for methodological rigor.
> The Bayesian model in Module 6 will explicitly account for this uncertainty.

---

### What this means for our analysis
Our core question shifts slightly but remains valid:

> *"Do years with higher conflict intensity correlate with higher inflation rates
> in Burkina Faso, Mali, and Niger, and does conflict in year T predict
> inflation in year T+1?"*

This is a **defensible, honest, and professionally rigorous** question given
the available data.

In [5]:
def fetch_wfp_prices(country: str, limit: int = 10000) -> pd.DataFrame:
    """
    Fetch WFP food price data from HDX API for a given country.
    Dataset: WFP Global Food Prices (resource ID stable since 2015)
    """
    url = "https://data.humdata.org/api/3/action/datastore_search"
    params = {
        "resource_id": "12d7c8e3-eff9-4db0-93b7-726825c4fe9a",
        "filters":     f'{{"adm0_name": "{country}"}}',
        "limit":       limit,
    }
    response = requests.get(url, params=params, timeout=30)
    data     = response.json()

    if not data.get("success"):
        print(f"  Failed to fetch data for {country}")
        return pd.DataFrame()

    records = data["result"]["records"]
    total   = data["result"]["total"]
    print(f"  {country}: {len(records):,} records fetched (total available: {total:,})")
    return pd.DataFrame(records)


# Fetch for all 3 countries
print("Downloading WFP food price data...\n")
frames = []
for country in ["Burkina Faso", "Mali", "Niger"]:
    df = fetch_wfp_prices(country)
    if not df.empty:
        frames.append(df)

df_wfp_raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal raw records : {len(df_wfp_raw):,}")
print(f"Columns           : {list(df_wfp_raw.columns)}")
print(f"\nSample:")
df_wfp_raw.head(3)


  Failed to fetch data for Burkina Faso
  Failed to fetch data for Mali
  Failed to fetch data for Niger


ValueError: No objects to concatenate

In [8]:
import io
# Direct CSV download, more reliable than the API
print("Downloading WFP Global Food Prices CSV...")

url = "https://data.humdata.org/dataset/wfp-food-prices/resource/12d7c8e3-eff9-4db0-93b7-726825c4fe9a/download/wfpvam_foodprices.csv"

response = requests.get(url, timeout=60)

if response.status_code == 200:
    df_wfp_raw = pd.read_csv(io.StringIO(response.text))
    print(f"Downloaded successfully!")
    print(f"Total records : {len(df_wfp_raw):,}")
    print(f"Columns       : {list(df_wfp_raw.columns)}")
    print(f"\nCountries available:")
    print(df_wfp_raw.iloc[:, 0].value_counts().head(20))
else:
    print(f"Failed: HTTP {response.status_code}")

Downloaded successfully!
Total records : 2,050,638
Columns       : ['adm0_id', 'adm0_name', 'adm1_id', 'adm1_name', 'mkt_id', 'mkt_name', 'cm_id', 'cm_name', 'cur_id', 'cur_name', 'pt_id', 'pt_name', 'um_id', 'um_name', 'mp_month', 'mp_year', 'mp_price', 'mp_commoditysource']

Countries available:
adm0_id
205.0    137746
115.0    137093
238.0    116588
196.0     82099
155.0     73843
116.0     72437
138.0     61188
43.0      60921
90.0      56971
181.0     54974
182.0     50285
68.0      47052
257.0     46053
270.0     42793
141.0     42784
170.0     42278
145.0     41207
29.0      39530
269.0     36806
42.0      35437
Name: count, dtype: int64


In [9]:
# Check exact country names for our 3 countries
print("Searching for our countries...\n")

for country in ["Burkina Faso", "Mali", "Niger", "burkina", "mali", "niger"]:
    matches = df_wfp_raw[
        df_wfp_raw["adm0_name"].str.contains(country, case=False, na=False)
    ]
    if len(matches) > 0:
        print(f"'{country}' -> {len(matches):,} records | exact name: '{matches['adm0_name'].iloc[0]}'")
    else:
        print(f"'{country}' -> NOT FOUND")

Searching for our countries...

'Burkina Faso' -> 35,437 records | exact name: 'Burkina Faso'
'Mali' -> 92,287 records | exact name: 'Mali'
'Niger' -> 105,259 records | exact name: 'Niger'
'burkina' -> 35,437 records | exact name: 'Burkina Faso'
'mali' -> 92,287 records | exact name: 'Mali'
'niger' -> 105,259 records | exact name: 'Niger'


In [10]:
# Filter for our 3 countries directly from the full CSV
COUNTRIES = ["Burkina Faso", "Mali", "Niger"]

df_wfp_raw_sahel = df_wfp_raw[
    df_wfp_raw["adm0_name"].isin(COUNTRIES)
].copy()

print(f"Total records for Sahel : {len(df_wfp_raw_sahel):,}")
print(f"\nBy country:")
print(df_wfp_raw_sahel["adm0_name"].value_counts().to_string())
print(f"\nDate range:")
print(f"  Year min : {df_wfp_raw_sahel['mp_year'].min()}")
print(f"  Year max : {df_wfp_raw_sahel['mp_year'].max()}")
print(f"\nCommodities available:")
print(df_wfp_raw_sahel["cm_name"].value_counts().head(20).to_string())

Total records for Sahel : 164,254

By country:
adm0_name
Mali            73843
Niger           54974
Burkina Faso    35437

Date range:
  Year min : 1990
  Year max : 2021

Commodities available:
cm_name
Millet - Retail                                  34193
Sorghum - Retail                                 20125
Rice (imported) - Retail                         19476
Maize - Retail                                   18369
Beans (niebe) - Retail                           18263
Rice (local) - Retail                            12984
Sorghum (white) - Retail                          6193
Groundnuts (shelled) - Retail                     6158
Maize (white) - Retail                            5759
Millet - Wholesale                                2487
Rice (imported) - Wholesale                       2456
Sorghum (local) - Wholesale                       2324
Groundnuts (unshelled) - Retail                   2183
Rice (paddy) - Retail                              981
Sugar - Retail            

In [11]:
# World Bank commodity prices API
# Pink Sheet — monthly commodity prices
url = "https://api.worldbank.org/v2/country/BFA;MLI;NER/indicator/FP.CPI.TOTL?format=json&per_page=500&mrv=60"

response = requests.get(url, timeout=30)
data     = response.json()

print(f"Status  : {response.status_code}")
print(f"Pages   : {data[0].get('pages')}")
print(f"Total   : {data[0].get('total')}")
print(f"\nSample:")
df = pd.DataFrame(data[1])
print(df[["country", "date", "value"]].head(10).to_string())

Status  : 200
Pages   : 1
Total   : 180

Sample:
                                 country  date       value
0  {'id': 'BF', 'value': 'Burkina Faso'}  2024  137.280900
1  {'id': 'BF', 'value': 'Burkina Faso'}  2023  131.759116
2  {'id': 'BF', 'value': 'Burkina Faso'}  2022  130.787482
3  {'id': 'BF', 'value': 'Burkina Faso'}  2021  114.434520
4  {'id': 'BF', 'value': 'Burkina Faso'}  2020  110.401266
5  {'id': 'BF', 'value': 'Burkina Faso'}  2019  108.359023
6  {'id': 'BF', 'value': 'Burkina Faso'}  2018  111.979765
7  {'id': 'BF', 'value': 'Burkina Faso'}  2017  109.831523
8  {'id': 'BF', 'value': 'Burkina Faso'}  2016  108.226524
9  {'id': 'BF', 'value': 'Burkina Faso'}  2015  107.751297


In [12]:


# IMF Data API, IHPC (CPI) for Burkina Faso, Mali, Niger
# Indicator: PCPI_IX = Consumer Price Index
countries = {
    "BF": "Burkina Faso",
    "ML": "Mali",
    "NE": "Niger",
}

frames = []
for code, name in countries.items():
    url = (
        f"https://www.imf.org/external/datamapper/api/v1/PCPI_IX/"
        f"{code}?periods=2020:2025"
    )
    r    = requests.get(url, timeout=30)
    data = r.json()
    print(f"{name}: status {r.status_code}")
    print(data)
    print()

Burkina Faso: status 200
{'api': {'version': '1', 'output-method': 'json'}}

Mali: status 200
{'api': {'version': '1', 'output-method': 'json'}}

Niger: status 200
{'api': {'version': '1', 'output-method': 'json'}}



In [13]:
# IMF Data API correct endpoint
# Indicator PCPI_IX = Consumer Price Index (monthly)

countries = {
    "BF": "Burkina Faso",
    "ML": "Mali",
    "NE": "Niger",
}

# Step 1: Check available indicators for Burkina Faso
url = "https://www.imf.org/external/datamapper/api/v1/indicators"
r   = requests.get(url, timeout=30)
print(f"Status: {r.status_code}")
data = r.json()

# Search for CPI / price related indicators
indicators = data.get("indicators", {})
print(f"Total indicators available: {len(indicators)}")
print("\nPrice/CPI related indicators:")
for key, val in indicators.items():
    label = val.get("label", "")
    if any(word in label.lower() for word in ["price", "cpi", "inflation", "consumer"]):
        print(f"  {key:<20} -> {label}")

Status: 200
Total indicators available: 133

Price/CPI related indicators:
  NGDPD                -> GDP, current prices
  NGDPDPC              -> GDP per capita, current prices

  PPPGDP               -> GDP, current prices
  PPPPC                -> GDP per capita, current prices
  PCPIPCH              -> Inflation rate, average consumer prices
  PCPIEPCH             -> Inflation rate, end of period consumer prices


AttributeError: 'NoneType' object has no attribute 'lower'

In [14]:

# Two useful indicators found:
# PCPIPCH  = Inflation rate, average consumer prices (annual % change)
# PCPIEPCH = Inflation rate, end of period consumer prices

countries = {
    "BF": "Burkina Faso",
    "ML": "Mali",
    "NE": "Niger",
}

frames = []
for code, name in countries.items():
    url = f"https://www.imf.org/external/datamapper/api/v1/PCPIPCH/{code}"
    r    = requests.get(url, timeout=30)
    data = r.json()

    # Navigate the response structure
    values = data.get("values", {}).get("PCPIPCH", {}).get(code, {})

    if not values:
        print(f"  {name}: no data found")
        continue

    # Convert to dataframe
    df = pd.DataFrame([
        {"country": name, "year": int(year), "inflation_pct": float(val)}
        for year, val in values.items()
        if 2018 <= int(year) <= 2025
    ])
    frames.append(df)
    print(f"  {name}: {len(df)} yearly records")

df_inflation = pd.concat(frames, ignore_index=True).sort_values(["country", "year"])

print(f"\nTotal records: {len(df_inflation):,}")
print(f"\nData:")
print(df_inflation.to_string(index=False))


  Burkina Faso: no data found
  Mali: no data found
  Niger: no data found


ValueError: No objects to concatenate

In [ ]:

# Inspect raw response structure
url  = "https://www.imf.org/external/datamapper/api/v1/PCPIPCH/BF"
r    = requests.get(url, timeout=30)
data = r.json()

print(f"Status  : {r.status_code}")
print(f"Keys    : {list(data.keys())}")
print(f"\nFull response:")
import json
print(json.dumps(data, indent=2))

Status  : 200
Keys    : ['values', 'api']

Full response:
{
  "values": {
    "PCPIPCH": {
      "SDN": {
        "1980": 26.5,
        "1981": 24,
        "1982": 26.6,
        "1983": 30.6,
        "1984": 33.7,
        "1985": 45.6,
        "1986": 60,
        "1987": 26.5,
        "1988": 62.9,
        "1989": 65.3,
        "1990": -0.9,
        "1991": 123.6,
        "1992": 117.6,
        "1993": 101.3,
        "1994": 115.5,
        "1995": 68.4,
        "1996": 86.8,
        "1997": 47.2,
        "1998": 24.6,
        "1999": 17.2,
        "2000": 7.1,
        "2001": 1.9,
        "2002": 22.2,
        "2003": 6.5,
        "2004": 9.7,
        "2005": 8.5,
        "2006": 7.2,
        "2007": 14.8,
        "2008": 14.3,
        "2009": 11.3,
        "2010": 13,
        "2011": 18.1,
        "2012": 35.6,
        "2013": 36.5,
        "2014": 36.9,
        "2015": 16.9,
        "2016": 17.8,
        "2017": 32.4,
        "2018": 63.3,
        "2019": 51,
        "2020": 163.3,
 

In [16]:
# Check what country codes IMF uses for our 3 countries
url  = "https://www.imf.org/external/datamapper/api/v1/PCPIPCH"
r    = requests.get(url, timeout=30)
data = r.json()

# List all available countries
countries_available = list(data.get("values", {}).get("PCPIPCH", {}).keys())
print(f"Total countries: {len(countries_available)}")
print(f"All codes: {sorted(countries_available)}")

# Search for our countries
print("\nSearching for Burkina, Mali, Niger...")
for code in countries_available:
    if code in ["BFA", "MLI", "NER", "BF", "ML", "NE", "BUR", "MAL", "NIG"]:
        values = data["values"]["PCPIPCH"][code]
        recent = {k: v for k, v in values.items() if int(k) >= 2020}
        print(f"  {code}: {recent}")

Total countries: 228
All codes: ['ABW', 'ADVEC', 'AFG', 'AFQ', 'AGO', 'ALB', 'AND', 'APQ', 'ARE', 'ARG', 'ARM', 'AS5', 'ATG', 'AUS', 'AUT', 'AZE', 'AZQ', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH', 'BLR', 'BLZ', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CAN', 'CAQ', 'CBQ', 'CHE', 'CHL', 'CHN', 'CIV', 'CMQ', 'CMR', 'COD', 'COG', 'COL', 'COM', 'CPV', 'CRI', 'CYP', 'CZE', 'DA', 'DEU', 'DJI', 'DMA', 'DNK', 'DOM', 'DZA', 'EAQ', 'ECU', 'EDE', 'EEQ', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'EU', 'EUQ', 'EURO', 'FIN', 'FJI', 'FRA', 'FSM', 'GAB', 'GBR', 'GEO', 'GHA', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD', 'GTM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KIR', 'KNA', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LCA', 'LIE', 'LKA', 'LSO', 'LTU', 'LUX', 'LVA', 'MAC', 'MAE', 'MAR', 'MDA', 'MDG', 'MDV', 'MECA', 'MEQ', 'MEX', 'MHL', 'MKD', 'MLI', 'MLT', 'MMR', 'MNE', 'MNG',

In [17]:
# Extract inflation data with correct country codes
country_codes = {
    "BFA": "Burkina Faso",
    "MLI": "Mali",
    "NER": "Niger",
}

# Get full data from previous response
url  = "https://www.imf.org/external/datamapper/api/v1/PCPIPCH"
r    = requests.get(url, timeout=30)
raw  = r.json()["values"]["PCPIPCH"]

frames = []
for code, name in country_codes.items():
    values = raw.get(code, {})
    df = pd.DataFrame([
        {"country": name, "year": int(y), "inflation_pct": float(v)}
        for y, v in values.items()
        # Only real data — exclude IMF projections (2025+)
        if 2018 <= int(y) <= 2024
    ])
    frames.append(df)
    print(f"  {name}: {len(df)} records")

df_inflation = pd.concat(frames, ignore_index=True).sort_values(["country", "year"]).reset_index(drop=True)

print(f"\nTotal records : {len(df_inflation):,}")
print(f"\nInflation data (real, 2018-2024):")
print(df_inflation.to_string(index=False))

  Burkina Faso: 7 records
  Mali: 7 records
  Niger: 7 records

Total records : 21

Inflation data (real, 2018-2024):
     country  year  inflation_pct
Burkina Faso  2018            2.0
Burkina Faso  2019           -3.2
Burkina Faso  2020            1.9
Burkina Faso  2021            3.9
Burkina Faso  2022           13.8
Burkina Faso  2023            0.9
Burkina Faso  2024            4.2
        Mali  2018            1.9
        Mali  2019           -3.0
        Mali  2020            0.5
        Mali  2021            3.8
        Mali  2022            9.7
        Mali  2023            2.1
        Mali  2024            3.2
       Niger  2018            2.8
       Niger  2019           -2.5
       Niger  2020            2.9
       Niger  2021            3.8
       Niger  2022            4.2
       Niger  2023            3.7
       Niger  2024            9.1


## 2. Building the Conflict-Inflation Dataset
---
We now aggregate ACLED conflict data at the **country × year** level
to match the IMF inflation data granularity.

**Conflict variables we compute:**
- `incidents`: total number of conflict events per year
- `fatalities`: total deaths per year
- `battles`: subset of events classified as armed battles
- `vac`: violence against civilians
- `locations`: number of unique locations affected
- `intensity_index`: a normalized conflict intensity score (0–100)

---

### Conflict Intensity Index, Min-Max Normalization

To make conflict intensity comparable across countries and years,
we normalize the incident count using **Min-Max normalization**:

$$I = \frac{x - x_{min}}{x_{max} - x_{min}} \times 100$$

Where:
- $x$ = number of incidents in a given country-year
- $x_{min}$ = minimum incidents across all countries and years
- $x_{max}$ = maximum incidents across all countries and years
- $I$ = normalized intensity score between **0** (least conflict) and **100** (most conflict)

**Example:**
| Country | Year | Incidents | Intensity Index |
|---|---|---|---|
| Niger | 2020 | 550 | ~0.0 |
| Mali | 2021 | 3,654 | ~52.3 |
| Burkina Faso | 2022 | 5,649 | ~100.0 |

>  **Note:** The index is computed globally across all countries 
> not per country. This means it captures relative conflict intensity
> across the entire Sahel region, not within each country individually.

**Join key:** `country` + `year`

In [18]:
# --- Annual conflict aggregates per country
acled_yearly = (
    df_acled[df_acled["year"].between(2018, 2024)]
    .groupby(["country", "year"])
    .agg(
        incidents  = ("event_id_cnty", "count"),
        fatalities = ("fatalities",    "sum"),
        battles    = ("event_type",    lambda x: (x == "Battles").sum()),
        vac        = ("event_type",    lambda x: (x == "Violence against civilians").sum()),
        locations  = ("location",      "nunique"),
    )
    .reset_index()
)

# --- Normalize incidents per year (conflict intensity index 0-100)
acled_yearly["intensity_index"] = (
    (acled_yearly["incidents"] - acled_yearly["incidents"].min()) /
    (acled_yearly["incidents"].max() - acled_yearly["incidents"].min()) * 100
).round(1)

print("ACLED yearly aggregates:")
print(acled_yearly.to_string(index=False))

ACLED yearly aggregates:
     country  year  incidents  fatalities  battles  vac  locations  intensity_index
Burkina Faso  2020        877        2304      243  337        424             19.2
Burkina Faso  2021       1868        2374      304  661        761             61.8
Burkina Faso  2022       2756        4246      515  637        964            100.0
Burkina Faso  2023       2360        8505      647  612        915             83.0
Burkina Faso  2024       1747        7493      606  386        744             56.6
        Mali  2020       1268        2856      424  437        528             36.0
        Mali  2021       1355        1913      327  498        570             39.7
        Mali  2022       1904        4863      474  626        835             63.4
        Mali  2023       2250        4347      467  754        979             78.2
        Mali  2024       1992        4023      418  643        756             67.1
       Niger  2020        550        1126      106 

## 3. Joining Conflict + Inflation Data
---

### Join strategy
We join ACLED conflict aggregates with IMF inflation data on `country` + `year`.

To test our core hypothesis, we build **two versions** of the joined dataset:

| Version | Description | Formula |
|---|---|---|
| **T+0** | Contemporaneous — conflict and inflation same year | $inflation(t) \sim incidents(t)$ |
| **T+1** | Lagged — conflict predicts next year's inflation | $inflation(t) \sim incidents(t-1)$ |

---

### The Lag Transformation

The T+1 lag is built by shifting conflict data **one year forward in time**:

$$x_{lag1}(t) = x(t-1)$$

Where:
- $x(t)$ = conflict variable (incidents, fatalities, intensity) in year $t$
- $x_{lag1}(t)$ = that same variable from year $t-1$, now associated with year $t$

**Concrete example:**

| Original data | After lag |
|---|---|
| Burkina 2020 → 550 incidents | `incidents_lag1` = 550 appears in **2021** |
| Burkina 2021 → 431 incidents | `incidents_lag1` = 431 appears in **2022** |
| Burkina 2022 → 5,649 incidents | `incidents_lag1` = 5,649 appears in **2023** |

**Why this matters economically:**
Supply chain disruptions, market closures, and trader behavior adjustments
do not materialize instantly in prices. A conflict shock in year T typically
propagates through the economy over the following months , making the
lagged relationship potentially stronger than the contemporaneous one.

>  The last year of conflict data (2024) has no corresponding
> inflation lag observation, so it will appear as `NaN` in `incidents_lag1`.
> This is expected and handled by dropping NaN values before correlation analysis.

# --- Join T+0 (contemporaneous)
df_joined = pd.merge(
    df_inflation,
    acled_yearly,
    on=["country", "year"],
    how="inner"
)

# --- Join T+1 (lagged: conflict at T predicts inflation at T+1)
acled_lag = acled_yearly.copy()
acled_lag["year"] = acled_lag["year"] + 1  # shift forward by 1 year
acled_lag = acled_lag.rename(columns={
    "incidents":       "incidents_lag1",
    "fatalities":      "fatalities_lag1",
    "battles":         "battles_lag1",
    "vac":             "vac_lag1",
    "intensity_index": "intensity_lag1",
    "locations":       "locations_lag1",
})

df_joined = pd.merge(
    df_joined,
    acled_lag[["country", "year",
               "incidents_lag1", "fatalities_lag1",
               "intensity_lag1"]],
    on=["country", "year"],
    how="left"
)

print(f"Joined dataset: {len(df_joined):,} records")
print(f"\nColumns: {list(df_joined.columns)}")
print(f"\nFull dataset:")
print(df_joined.to_string(index=False))

## 4. Correlation Analysis
---
We measure the Pearson correlation between conflict variables
and inflation, with and without the 1-year lag.

**Interpretation guide:**
- `r > 0.6` → Strong positive correlation
- `r 0.3–0.6` → Moderate correlation
- `r < 0.3` → Weak correlation
- `p < 0.05` → Statistically significant

>  With only 7 data points per country (2018–2024), statistical
> power is limited. Results are **exploratory**, not conclusive.
> The Bayesian model in Module 6 will quantify this uncertainty explicitly.

In [22]:
from scipy import stats

print("=" * 60)
print("CORRELATION: CONFLICT → INFLATION")
print("=" * 60)

pairs = {
    "Incidents (T+0)":        ("incidents",       "inflation_pct"),
    "Fatalities (T+0)":       ("fatalities",      "inflation_pct"),
    "Intensity index (T+0)":  ("intensity_index", "inflation_pct"),
    "Incidents (T+1 lag)":    ("incidents_lag1",  "inflation_pct"),
    "Fatalities (T+1 lag)":   ("fatalities_lag1", "inflation_pct"),
    "Intensity index (T+1)":  ("intensity_lag1",  "inflation_pct"),
}

results = []
for label, (x_col, y_col) in pairs.items():
    subset = df_joined.dropna(subset=[x_col, y_col])
    if len(subset) < 4:
        continue
    r, p = stats.pearsonr(subset[x_col], subset[y_col])
    results.append({
        "Variable":    label,
        "N":           len(subset),
        "Pearson r":   round(r, 3),
        "p-value":     round(p, 4),
        "Strength":    "Strong" if abs(r) > 0.6 else "Moderate" if abs(r) > 0.3 else "Weak",
        "Significant": "True" if p < 0.05 else "False",
    })

df_corr = pd.DataFrame(results)
print(df_corr.to_string(index=False))

CORRELATION: CONFLICT → INFLATION
             Variable  N  Pearson r  p-value Strength Significant
      Incidents (T+0) 15      0.284   0.3056     Weak       False
     Fatalities (T+0) 15     -0.001   0.9976     Weak       False
Intensity index (T+0) 15      0.284   0.3051     Weak       False
  Incidents (T+1 lag) 12     -0.159   0.6227     Weak       False
 Fatalities (T+1 lag) 12     -0.321   0.3086 Moderate       False
Intensity index (T+1) 12     -0.159   0.6222     Weak       False


## 6. Interpretation of Results
---

### Key finding
No statistically significant linear correlation was found between
conflict intensity and national inflation rates at the country-year level.

| Metric | Value | Interpretation |
|---|---|---|
| Best r (T+0) | 0.284 | Weak positive, not significant |
| Best r (T+1) | -0.321 | Weak negative, not significant |
| All p-values | > 0.05 | Cannot reject null hypothesis |

### Why this result is methodologically expected

**1. Statistical power is limited (N=15)**
With 5 years × 3 countries, we have insufficient observations for
robust inference. A minimum of N=30 is generally required.

**2. Aggregation mismatch**
National CPI captures systemic shocks: global oil prices, BCEAO
monetary policy, droughts — that dilute the conflict signal.
A conflict in northern Burkina Faso does not necessarily move
the national CPI, but it does move **local market prices**.

**3. The relationship may be non-linear**
Conflict effects on prices may follow a threshold dynamic:
- Low conflict → no price effect
- High conflict → sharp price spike

A linear correlation coefficient cannot capture this.

### What this means for Module 6
This result is precisely why we need the **Bayesian model**:

- It will work at the **sub-national level** (admin1) where the
  relationship is more direct
- It will **quantify uncertainty** explicitly rather than rely on
  binary significance tests
- It will **control for confounders** (rainfall, global prices)
- It will capture **non-linear threshold effects**

> This analysis is honest and reproducible. Showing null results
> with a clear methodological explanation is more valuable than
> forcing a spurious correlation.

## 5. Visualizations
---
Three charts to communicate the conflict-inflation relationship:

1. **Scatter plot**: conflict intensity vs inflation per country per year
2. **Dual axis chart**: incidents and inflation on the same timeline
3. **Lag comparison**: T+0 vs T+1 correlation strength

In [27]:

COUNTRY_COLORS = {
    "Burkina Faso": "#e74c3c",
    "Mali":         "#3498db",
    "Niger":        "#2ecc71",
}

# --- Plot 1: Scatter, conflict intensity vs inflation
fig1 = px.scatter(
    df_joined,
    x="incidents",
    y="inflation_pct",
    color="country",
    size="fatalities",
    text="year",
    title="Conflict Intensity vs Inflation Rate: Sahel 2020–2024",
    labels={
        "incidents":     "Annual Conflict Incidents",
        "inflation_pct": "Inflation Rate (%)",
        "country":       "Country",
    },
    color_discrete_map=COUNTRY_COLORS,
    template="plotly_dark",
    trendline="ols",
)
fig1.update_traces(textposition="top center", textfont_size=10)
fig1.update_layout(height=500)
fig1.show()

# --- Plot 2: Dual axis, incidents (bars) + inflation (line) per country
fig2 = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=list(COUNTRY_COLORS.keys()),
    vertical_spacing=0.10,
    specs=[
        [{"secondary_y": True}],
        [{"secondary_y": True}],
        [{"secondary_y": True}],
    ]
)

for i, (country, color) in enumerate(COUNTRY_COLORS.items(), start=1):
    sub = df_joined[df_joined["country"] == country].sort_values("year")

    # Incidents — primary Y axis (left)
    fig2.add_trace(go.Bar(
        x=sub["year"],
        y=sub["incidents"],
        name="Incidents",
        marker_color=color,
        opacity=0.7,
        showlegend=(i == 1),
        legendgroup="incidents",
    ), row=i, col=1, secondary_y=False)

    # Inflation: secondary Y axis (right)
    fig2.add_trace(go.Scatter(
        x=sub["year"],
        y=sub["inflation_pct"],
        name="Inflation %",
        mode="lines+markers",
        line=dict(color="#f1c40f", width=2),
        marker=dict(size=8),
        showlegend=(i == 1),
        legendgroup="inflation",
    ), row=i, col=1, secondary_y=True)

    # Left axis label
    fig2.update_yaxes(
        title_text="Incidents",
        row=i, col=1,
        secondary_y=False,
        title_font=dict(color=color),
    )

    # Right axis label
    fig2.update_yaxes(
        title_text="Inflation %",
        row=i, col=1,
        secondary_y=True,
        title_font=dict(color="#f1c40f"),
    )

fig2.update_layout(
    title="Conflict Incidents vs Inflation Rate by Country — 2018–2024",
    template="plotly_dark",
    height=750,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig2.show()

# --- Plot 3: Lag comparison bar chart
fig3 = px.bar(
    df_corr,
    x="Variable",
    y="Pearson r",
    color="Significant",
    title="Correlation Strength: Conflict → Inflation (T+0 vs T+1 lag)",
    labels={"Pearson r": "Pearson Correlation Coefficient"},
    color_discrete_map={"✅": "#2ecc71", "❌": "#e74c3c"},
    template="plotly_dark",
    text="Pearson r",
)
fig3.add_hline(
    y=0.6, line_dash="dash", line_color="#f1c40f",
    annotation_text="Strong threshold (0.6)",
    annotation_position="top right",
)
fig3.add_hline(
    y=0.3, line_dash="dash", line_color="#3498db",
    annotation_text="Moderate threshold (0.3)",
    annotation_position="top right",
)
fig3.add_hline(y=-0.3, line_dash="dash", line_color="#3498db")
fig3.add_hline(y=-0.6, line_dash="dash", line_color="#f1c40f")
fig3.update_traces(textposition="outside")
fig3.update_layout(height=450)
fig3.show()

In [24]:
# Save for dashboard integration
df_joined.to_csv("../data/processed/conflict_inflation_joined.csv", index=False)
df_corr.to_csv("../data/processed/correlation_results.csv",         index=False)
acled_yearly.to_csv("../data/processed/acled_yearly.csv",           index=False)

print("Module 5 data saved:")
print("  -> data/processed/conflict_inflation_joined.csv")
print("  -> data/processed/correlation_results.csv")
print("  -> data/processed/acled_yearly.csv")
print("\nModule 5 complete. Ready for Bayesian modeling (Module 6).")

Module 5 data saved:
  -> data/processed/conflict_inflation_joined.csv
  -> data/processed/correlation_results.csv
  -> data/processed/acled_yearly.csv

Module 5 complete. Ready for Bayesian modeling (Module 6).
